In [0]:
%sql
DROP TABLE IF EXISTS workspace_1.silver.crm_products 

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

# Reading From Bronze

In [0]:
df= spark.table("workspace_1.bronze.crm_prd_info")

# Data Transformation

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Product Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

# Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

# Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)
     

# Date Casting

In [0]:
# Cast to DateType first
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))
df = df.withColumn("prd_end_dt", col("prd_end_dt").cast(DateType()))

# Fix inverted dates: swap when end_date < start_date
# Store originals in temp columns
df = df.withColumn("_temp_start", col("prd_start_dt"))
df = df.withColumn("_temp_end", col("prd_end_dt"))

# Swap dates when end_date is before start_date
df = df.withColumn(
    "prd_start_dt",
    F.when(
        (col("_temp_end").isNotNull()) & (col("_temp_end") < col("_temp_start")),
        col("_temp_end")  # Use end as start
    ).otherwise(col("_temp_start"))  # Keep original start
)

df = df.withColumn(
    "prd_end_dt",
    F.when(
        (col("_temp_end").isNotNull()) & (col("_temp_end") < col("_temp_start")),
        col("_temp_start")  # Use start as end
    ).otherwise(col("_temp_end"))  # Keep original end
)

# Drop temp columns
df = df.drop("_temp_start", "_temp_end")

In [0]:
# Add is_current flag to identify current products (those with NULL end_date)
df = df.withColumn("is_current", col("prd_end_dt").isNull())

# Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
# Check for records where end_date is before start_date
invalid_dates = df.filter(
    (col("end_date").isNotNull()) & 
    (col("end_date") < col("start_date"))
)

invalid_count = invalid_dates.count()
print(f"Found {invalid_count} records with end_date before start_date")

if invalid_count > 0:
    print("\nSample of invalid records:")
    invalid_dates.select("product_id", "product_number", "product_name", "start_date", "end_date").limit(10).display()

# Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace_1.silver.crm_products")

In [0]:
df.display()

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace_1.silver.crm_products LIMIT 10